In [1]:
import os
import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, auc
)
from sklearn.preprocessing import label_binarize

# Paths (assuming notebook is in the 'training' folder)
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_DIR = os.path.join(BASE_DIR, 'data', 'processed')
SELECTED_DIR = os.path.join(DATA_DIR, 'selected')
MODELS_DIR = os.path.join(BASE_DIR, 'models')
RESULTS_DIR = os.path.join(BASE_DIR, 'results')

# Create directories
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Class names
CLASS_NAMES = ['NORM', 'MI', 'STTC', 'CD', 'HYP']

In [2]:
def load_data(feature_set='top100'):
    """Load preprocessed data for a specific feature set."""
    print(f'Loading {feature_set} feature set...')
    
    if feature_set == 'full':
        X_train = pd.read_csv(os.path.join(DATA_DIR, 'X_train.csv'))
        X_val = pd.read_csv(os.path.join(DATA_DIR, 'X_val.csv'))
        X_test = pd.read_csv(os.path.join(DATA_DIR, 'X_test.csv'))
    else:
        # Load selected features
        suffix = feature_set.replace('top', '_top')
        X_train = pd.read_csv(os.path.join(SELECTED_DIR, f'X_train{suffix}.csv'))
        X_val = pd.read_csv(os.path.join(SELECTED_DIR, f'X_val{suffix}.csv'))
        X_test = pd.read_csv(os.path.join(SELECTED_DIR, f'X_test{suffix}.csv'))
    
    y_train = pd.read_csv(os.path.join(DATA_DIR, 'y_train.csv'))['label']
    y_val = pd.read_csv(os.path.join(DATA_DIR, 'y_val.csv'))['label']
    y_test = pd.read_csv(os.path.join(DATA_DIR, 'y_test.csv'))['label']
    
    # Load class weights
    class_weights_df = pd.read_csv(os.path.join(DATA_DIR, 'class_weights.csv'))
    class_weights = dict(zip(class_weights_df.columns, class_weights_df.iloc[0].values))
    
    print(f'  Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')
    
    return X_train, X_val, X_test, y_train, y_val, y_test, class_weights

In [3]:
def get_models_and_params(class_weights):
    """Define models and their hyperparameter grids."""
    
    models = {
        'DecisionTree': {
            'model': DecisionTreeClassifier(random_state=42, class_weight=class_weights),
            'params': {
                'max_depth': [5, 10, 15, 20, None],
                'min_samples_split': [2, 5, 10],
                'criterion': ['gini', 'entropy'],
                'min_samples_leaf': [1, 2, 4]
            }
        },
        'RandomForest': {
            'model': RandomForestClassifier(random_state=42, class_weight=class_weights, n_jobs=-1),
            'params': {
                'n_estimators': [100, 200],
                'max_depth': [10, 20, None],
                'min_samples_split': [2, 5],
                'min_samples_leaf': [1, 2],
                'max_features': ['sqrt', 'log2']
            }
        },
        'NaiveBayes': {
            'model': GaussianNB(),
            'params': {
                'var_smoothing': [1e-11, 1e-10, 1e-9, 1e-8, 1e-7]
            }
        },
        'SVM': {
            'model': SVC(random_state=42, class_weight=class_weights, probability=True),
            'params': {
                'C': [0.1, 1, 10],
                'kernel': ['linear', 'rbf'],
                'gamma': ['scale', 'auto']
            }
        }
    }
    
    return models

In [4]:
def train_model_with_grid_search(model_name, model_config, X_train, y_train, X_val, y_val):
    """Train a model using grid search for hyperparameter tuning."""
    
    print(f'\nTraining {model_name}...')
    
    # Combine train and validation for cross-validation
    X_combined = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
    y_combined = pd.concat([y_train, y_val], axis=0).reset_index(drop=True)
    
    # Stratified 5-fold CV
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    start_time = time.time()
    
    grid_search = GridSearchCV(
        model_config['model'],
        model_config['params'],
        cv=cv,
        scoring='f1_macro',
        n_jobs=-1,
        verbose=1,
        refit=True
    )
    
    grid_search.fit(X_combined, y_combined)
    
    training_time = time.time() - start_time
    
    print(f'  Best parameters: {grid_search.best_params_}')
    print(f'  Best CV score (macro F1): {grid_search.best_score_:.4f}')
    print(f'  Training time: {training_time:.2f}s')
    
    return grid_search.best_estimator_, grid_search.best_params_, training_time

In [5]:
def evaluate_model(model, X_test, y_test, model_name):
    """Evaluate a trained model on the test set."""
    
    print(f'\nEvaluating {model_name} on test set...')
    
    y_pred = model.predict(X_test)
    
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_test)
    else:
        y_prob = None
    
    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'macro_f1': f1_score(y_test, y_pred, average='macro'),
        'weighted_f1': f1_score(y_test, y_pred, average='weighted'),
        'macro_precision': precision_score(y_test, y_pred, average='macro'),
        'macro_recall': recall_score(y_test, y_pred, average='macro'),
    }
    
    class_report = classification_report(y_test, y_pred, output_dict=True)
    cm = confusion_matrix(y_test, y_pred, labels=CLASS_NAMES)
    
    if y_prob is not None:
        try:
            y_test_bin = label_binarize(y_test, classes=CLASS_NAMES)
            roc_auc = roc_auc_score(y_test_bin, y_prob, average='macro', multi_class='ovr')
            metrics['roc_auc'] = roc_auc
        except Exception as e:
            print(f'  Warning: Could not compute ROC-AUC: {e}')
            metrics['roc_auc'] = None
    else:
        metrics['roc_auc'] = None
    
    print(f'  Accuracy: {metrics["accuracy"]:.4f}')
    print(f'  Macro F1: {metrics["macro_f1"]:.4f}')
    print(f'  Weighted F1: {metrics["weighted_f1"]:.4f}')
    if metrics['roc_auc']:
        print(f'  ROC-AUC: {metrics["roc_auc"]:.4f}')
    
    return metrics, class_report, cm, y_pred, y_prob

In [6]:
def plot_confusion_matrix(cm, model_name, feature_set, save_path):
    """Plot and save confusion matrix."""
    
    plt.figure(figsize=(8, 6))
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                vmin=0, vmax=1)
    for i in range(len(CLASS_NAMES)):
        for j in range(len(CLASS_NAMES)):
            plt.text(j + 0.5, i + 0.75, f'({cm[i, j]})',
                     ha='center', va='center', fontsize=8, color='gray')
    plt.title(f'Confusion Matrix: {model_name} ({feature_set})\n(normalized, with counts)',
              fontsize=12, fontweight='bold')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()

In [7]:
def plot_roc_curves(y_test, y_prob, model_name, feature_set, save_path):
    """Plot ROC curves for each class."""
    
    if y_prob is None:
        return
    
    y_test_bin = label_binarize(y_test, classes=CLASS_NAMES)
    n_classes = len(CLASS_NAMES)
    plt.figure(figsize=(10, 8))
    colors = plt.cm.Set1(np.linspace(0, 1, n_classes))
    for i, (class_name, color) in enumerate(zip(CLASS_NAMES, colors)):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_prob[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, color=color, lw=2,
                 label=f'{class_name} (AUC = {roc_auc:.3f})')
    plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title(f'ROC Curves: {model_name} ({feature_set})', fontsize=13, fontweight='bold')
    plt.legend(loc='lower right')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()

In [8]:
def save_results(all_results, feature_set):
    """Save results to CSV for report."""
    
    summary_data = []
    for model_name, result in all_results.items():
        row = {
            'Model': model_name,
            'Feature Set': feature_set,
            'Accuracy': result['metrics']['accuracy'],
            'Macro F1': result['metrics']['macro_f1'],
            'Weighted F1': result['metrics']['weighted_f1'],
            'Macro Precision': result['metrics']['macro_precision'],
            'Macro Recall': result['metrics']['macro_recall'],
            'ROC-AUC': result['metrics'].get('roc_auc'),
            'Training Time (s)': result['training_time']
        }
        summary_data.append(row)
    summary_df = pd.DataFrame(summary_data)
    summary_df.to_csv(os.path.join(RESULTS_DIR, f'summary_{feature_set}.csv'), index=False)
    
    for model_name, result in all_results.items():
        class_report = result['class_report']
        class_data = []
        for class_name in CLASS_NAMES:
            if class_name in class_report:
                row = {
                    'Class': class_name,
                    'Precision': class_report[class_name]['precision'],
                    'Recall': class_report[class_name]['recall'],
                    'F1-Score': class_report[class_name]['f1-score'],
                    'Support': class_report[class_name]['support']
                }
                class_data.append(row)
        class_df = pd.DataFrame(class_data)
        class_df.to_csv(
            os.path.join(RESULTS_DIR, f'class_metrics_{model_name}_{feature_set}.csv'),
            index=False
        )
    print(f'\nResults saved to {RESULTS_DIR}')

In [9]:
def create_comparison_plots(all_results_by_feature):
    """Create comparison plots across models and feature sets."""
    
    comparison_data = []
    for feature_set, results in all_results_by_feature.items():
        for model_name, result in results.items():
            comparison_data.append({
                'Model': model_name,
                'Feature Set': feature_set,
                'Accuracy': result['metrics']['accuracy'],
                'Macro F1': result['metrics']['macro_f1'],
                'Training Time': result['training_time']
            })
    df = pd.DataFrame(comparison_data)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ax1 = axes[0]
    x = np.arange(len(['top50', 'top100', 'top200']))
    width = 0.2
    for i, model in enumerate(['DecisionTree', 'RandomForest', 'NaiveBayes', 'SVM']):
        model_data = df[df['Model'] == model].sort_values('Feature Set')
        ax1.bar(x + i*width, model_data['Macro F1'], width, label=model, alpha=0.8)
    ax1.set_xlabel('Feature Set')
    ax1.set_ylabel('Macro F1 Score')
    ax1.set_title('Model Comparison: Macro F1 Score', fontweight='bold')
    ax1.set_xticks(x + 1.5*width)
    ax1.set_xticklabels(['Top 50', 'Top 100', 'Top 200'])
    ax1.legend()
    ax1.grid(axis='y', alpha=0.3)
    ax1.set_ylim(0, 1)
    ax2 = axes[1]
    for i, model in enumerate(['DecisionTree', 'RandomForest', 'NaiveBayes', 'SVM']):
        model_data = df[df['Model'] == model].sort_values('Feature Set')
        ax2.bar(x + i*width, model_data['Training Time'], width, label=model, alpha=0.8)
    ax2.set_xlabel('Feature Set')
    ax2.set_ylabel('Training Time (seconds)')
    ax2.set_title('Model Comparison: Training Time', fontweight='bold')
    ax2.set_xticks(x + 1.5*width)
    ax2.set_xticklabels(['Top 50', 'Top 100', 'Top 200'])
    ax2.legend()
    ax2.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'model_comparison.png'), dpi=300, bbox_inches='tight')
    plt.close()
    print('✓ Saved model comparison plots')

In [10]:
def main(force_retrain: bool = False):
    """Main training and evaluation pipeline.

    If ``force_retrain`` is False, existing saved models are loaded from disk
    (if available) instead of being retrained. This helps avoid rerunning
    expensive grid searches when you rerun the notebook.
    """
    
    print("=" * 60)
    print("ECG Classification Model Training")
    print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 60)
    
    feature_sets = ['top50', 'top100', 'top200']
    all_results_by_feature = {}
    
    for feature_set in feature_sets:
        print("\n" + "=" * 60)
        print(f"Processing feature set: {feature_set.upper()}")
        print("=" * 60)
        
        X_train, X_val, X_test, y_train, y_val, y_test, class_weights = load_data(feature_set)
        models = get_models_and_params(class_weights)
        all_results = {}
        
        for model_name, model_config in models.items():
            model_path = os.path.join(MODELS_DIR, f"{model_name}_{feature_set}.pkl")
            best_params = None
            
            if (not force_retrain) and os.path.exists(model_path):
                print(f"  Found existing model for {model_name} ({feature_set}), loading from {model_path}...")
                with open(model_path, "rb") as f:
                    trained_model = pickle.load(f)
                training_time = float("nan")
            else:
                trained_model, best_params, training_time = train_model_with_grid_search(
                    model_name, model_config, X_train, y_train, X_val, y_val
                )
                with open(model_path, 'wb') as f:
                    pickle.dump(trained_model, f)
                print(f"  Model saved to {model_path}")
            
            metrics, class_report, cm, y_pred, y_prob = evaluate_model(
                trained_model, X_test, y_test, model_name
            )
            
            all_results[model_name] = {
                'model': trained_model,
                'best_params': best_params,
                'training_time': training_time,
                'metrics': metrics,
                'class_report': class_report,
                'confusion_matrix': cm,
                'predictions': y_pred,
                'probabilities': y_prob
            }
            
            cm_path = os.path.join(RESULTS_DIR, f"confusion_matrix_{model_name}_{feature_set}.png")
            plot_confusion_matrix(cm, model_name, feature_set, cm_path)
            
            roc_path = os.path.join(RESULTS_DIR, f"roc_curves_{model_name}_{feature_set}.png")
            plot_roc_curves(y_test, y_prob, model_name, feature_set, roc_path)
        
        save_results(all_results, feature_set)
        all_results_by_feature[feature_set] = all_results
    
    create_comparison_plots(all_results_by_feature)
    
    print("=" * 60)
    print("FINAL SUMMARY")
    print("=" * 60)
    
    best_score = 0.0
    best_model = None
    best_feature = None
    
    for feature_set, results in all_results_by_feature.items():
        print(f"\n{feature_set.upper()}:")
        for model_name, result in results.items():
            macro_f1 = result['metrics']['macro_f1']
            acc = result['metrics']['accuracy']
            print(f"  {model_name}: Macro F1 = {macro_f1:.4f}, Accuracy = {acc:.4f}")
            if macro_f1 > best_score:
                best_score = macro_f1
                best_model = model_name
                best_feature = feature_set
    
    print(f"\n*** Best model: {best_model} with {best_feature} features ***")
    print(f"*** Macro F1 Score: {best_score:.4f} ***")
    print("=" * 60)
    print(f"Completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 60)

In [11]:
main(   )

ECG Classification Model Training
Started at: 2025-12-16 02:43:39

Processing feature set: TOP50
Loading top50 feature set...
  Train: (17084, 50), Val: (2146, 50), Test: (2158, 50)
  Found existing model for DecisionTree (top50), loading from c:\Users\Pc\Desktop\LOCALS\ML_PTB-XL\PTB-XL-Heart-Anomaly-Diagnosis-Project\models\DecisionTree_top50.pkl...

Evaluating DecisionTree on test set...
  Accuracy: 0.6622
  Macro F1: 0.5823
  Weighted F1: 0.6669
  ROC-AUC: 0.4570

Training RandomForest...
Fitting 5 folds for each of 48 candidates, totalling 240 fits
  Best parameters: {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 200}
  Best CV score (macro F1): 0.6880
  Training time: 194.00s
  Model saved to c:\Users\Pc\Desktop\LOCALS\ML_PTB-XL\PTB-XL-Heart-Anomaly-Diagnosis-Project\models\RandomForest_top50.pkl

Evaluating RandomForest on test set...
  Accuracy: 0.7377
  Macro F1: 0.6443
  Weighted F1: 0.7295
  ROC-AUC: 0.4631
  Found exi